# Evaluate SAC on Fixed Obstacles

Load the saved SAC checkpoint and evaluate it on the same fixed unseen-obstacle configuration used for DDPG.

## Imports and Paths

In [1]:
from pathlib import Path
import sys

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()
    if BASE_DIR.name != "Continuous_Diff_Drive":
        BASE_DIR = BASE_DIR / "Continuous_Diff_Drive"

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from diff_drive_agent import DiffDriveSACAgent
from diff_drive_env import DiffDriveEnv

BASE_DIR

PosixPath('/Users/anthonymc/Desktop/NAML_project/Continuous_Diff_Drive')

## Fixed Evaluation Configuration

In [2]:
CHECKPOINT_PATH = BASE_DIR / "models" / "sac_checkpoint.pt"
EVALUATION_VIDEO_DIR = BASE_DIR / "videos" / "evaluation_sac"
FIXED_EVALUATION_NAME_PREFIX = "sac_diff_drive_eval_unseen_fixed_obstacles_greedy"
RANDOM_EVALUATION_NAME_PREFIX = "sac_diff_drive_eval_random_obstacles_greedy"

# A room with a few axis-aligned rectangular obstacles.
# Each obstacle is (x, y, width, height) in metres, origin at bottom-left.
OBSTACLES = [
    (2.0, 4.0, 3.0, 0.3),
    (5.0, 2.0, 0.3, 3.0),
    (7.0, 6.0, 1.5, 0.3),
]

ENV_KWARGS = dict(
    room_size       = (10.0, 10.0),
    obstacles       = OBSTACLES,
    random_obst     = False,
    robot_start     = (1.0, 1.0),
    goal_pos        = (8.5, 8.5),
    max_step        = 1000,
    n_lidar_rays    = 16,
    lidar_max_range = 5.0,
    robot_radius    = 0.3,
    dt              = 0.1,
    render_mode     = "rgb_array",
)

ENV_KWARGS_RANDOM = dict(
    room_size       = (10.0, 10.0),
    obstacles       = None,
    random_obst     = True,
    robot_start     = (1.0, 1.0),
    goal_pos        = (8.5, 8.5),
    max_step        = 1000,
    n_lidar_rays    = 16,
    lidar_max_range = 5.0,
    robot_radius    = 0.3,
    dt              = 0.1,
    render_mode     = "rgb_array",
    obstacle_mode   = "curriculum",
)

ACTOR_LR      = 3e-4
CRITIC_LR     = 3e-4
ALPHA_LR      = 3e-4
DISCOUNT      = 0.99
TAU           = 0.005
BATCH_SIZE    = 256
BUFFER_SIZE   = 100_000
HIDDEN_DIM    = 64
WARMUP_STEPS  = 5_000
DEVICE        = "cpu"
N_EPISODES    = 3
N_RANDOM_EPISODES = 10

EVALUATION_VIDEO_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH

PosixPath('/Users/anthonymc/Desktop/NAML_project/Continuous_Diff_Drive/models/sac_checkpoint.pt')

## Environment and Agent

In [3]:
env = DiffDriveEnv(**ENV_KWARGS)
env_random = DiffDriveEnv(**ENV_KWARGS_RANDOM)

agent = DiffDriveSACAgent(
    env          = env,
    actor_lr     = ACTOR_LR,
    critic_lr    = CRITIC_LR,
    alpha_lr     = ALPHA_LR,
    discount     = DISCOUNT,
    tau          = TAU,
    batch_size   = BATCH_SIZE,
    buffer_size  = BUFFER_SIZE,
    hidden_dim   = HIDDEN_DIM,
    warmup_steps = WARMUP_STEPS,
    device       = DEVICE,
)

agent_random = DiffDriveSACAgent(
    env          = env_random,
    actor_lr     = ACTOR_LR,
    critic_lr    = CRITIC_LR,
    alpha_lr     = ALPHA_LR,
    discount     = DISCOUNT,
    tau          = TAU,
    batch_size   = BATCH_SIZE,
    buffer_size  = BUFFER_SIZE,
    hidden_dim   = HIDDEN_DIM,
    warmup_steps = WARMUP_STEPS,
    device       = DEVICE,
)

## Load Checkpoint

In [4]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"No checkpoint found at {CHECKPOINT_PATH}")

agent.load_checkpoint(CHECKPOINT_PATH, load_optimizers=False)
agent_random.load_checkpoint(CHECKPOINT_PATH, load_optimizers=False)

SAC checkpoint loaded from /Users/anthonymc/Desktop/NAML_project/Continuous_Diff_Drive/models/sac_checkpoint.pt
SAC checkpoint loaded from /Users/anthonymc/Desktop/NAML_project/Continuous_Diff_Drive/models/sac_checkpoint.pt


{'actor': OrderedDict([('action_scale', tensor([0.7500, 3.1416])),
              ('action_bias', tensor([0.2500, 0.0000])),
              ('base_net.0.weight',
               tensor([[-1.2159, -0.2717, -0.3675,  ..., -0.0860,  0.8484,  0.9880],
                       [-0.4893, -0.0337, -0.8873,  ...,  0.7808, -0.0345, -0.1625],
                       [-0.3793, -0.4696,  0.1109,  ..., -0.6140, -0.3956,  2.5506],
                       ...,
                       [ 0.4128,  0.3348,  0.1505,  ...,  0.0433, -0.0546, -2.6852],
                       [-0.2486, -0.3660, -0.3364,  ...,  0.5776, -0.0892, -1.5072],
                       [ 0.4065,  0.5038,  0.3083,  ..., -0.5357, -0.8085, -0.8669]])),
              ('base_net.0.bias',
               tensor([ 0.2353,  0.1295, -0.3910,  0.0457,  0.3668, -0.3337, -0.3014, -0.1218,
                        0.1816, -0.0950,  0.3142, -0.1433,  0.0977,  0.1830,  0.2120, -0.0616,
                        0.0000, -0.0067, -0.5072,  0.1265, -0.1713,  0.2352

## Greedy Evaluation

In [5]:
agent.eval_recorded(
    video_folder = EVALUATION_VIDEO_DIR,
    name_prefix  = FIXED_EVALUATION_NAME_PREFIX,
    n_episodes   = N_EPISODES,
)

/Users/anthonymc/Desktop/NAML_project/venv/lib/python3.11/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /Users/anthonymc/Desktop/NAML_project/Continuous_Diff_Drive/videos/evaluation_sac folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


  SAC eval ep 1: reward = 591.2 | steps = 127 | goal reached: yes


KeyboardInterrupt: 

## Evaluation Videos

In [ ]:
from IPython.display import Video, display

video_paths = sorted(EVALUATION_VIDEO_DIR.glob(f"{FIXED_EVALUATION_NAME_PREFIX}*.mp4"))

if not video_paths:
    print(f"No videos found yet in {EVALUATION_VIDEO_DIR}")

for video_path in video_paths:
    print(video_path.name)
    display(Video(filename=str(video_path), embed=True))

## Random Obstacle Evaluation

In [ ]:
agent_random.eval_recorded(
    video_folder = EVALUATION_VIDEO_DIR,
    name_prefix  = RANDOM_EVALUATION_NAME_PREFIX,
    n_episodes   = N_RANDOM_EPISODES,
)

In [ ]:
video_paths = sorted(EVALUATION_VIDEO_DIR.glob(f"{RANDOM_EVALUATION_NAME_PREFIX}*.mp4"))

if not video_paths:
    print(f"No videos found yet in {EVALUATION_VIDEO_DIR}")

for video_path in video_paths:
    print(video_path.name)
    display(Video(filename=str(video_path), embed=True))